# 机器学习数学基础：从斜率到反向传播

这本笔记不假设你已经熟悉多元微积分。目标不是背公式，而是建立三个核心直觉：

1. **导数**：输入稍微改变时，输出会怎样改变？
2. **梯度**：有多个输入时，往哪个方向改变最快？
3. **链式法则**：复杂函数由多步计算组成时，影响如何一层层传递？

## 学习路线

`函数与图像 → 单变量导数 → 多变量函数 → 偏导数 → 梯度 → 梯度下降 → Jacobian → Hessian → 反向传播`

## 符号阅读

- $x$、$y$：标量，即一个数；
- $\mathbf{x}$：向量，即一列或一组数；
- $X$：矩阵；
- $\frac{dy}{dx}$：$y$ 对 $x$ 的导数；
- $\frac{\partial f}{\partial x}$：多变量函数对 $x$ 的偏导数；
- $\nabla f$：由全部偏导数组成的梯度向量。

> 每看到一个公式，先问：输入是什么？输出是什么？每个量的 shape 是什么？


In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


# 第一部分：单变量函数与导数

## 1. 函数是什么？

函数可以理解为一台机器：

`输入 x → 按规则计算 → 输出 y`

例如 $f(x)=x^2$。输入 3，输出 9。机器学习模型也是函数，只是输入、参数和计算过程更复杂：

$$
\hat y=f_{\mathbf w,b}(\mathbf x).
$$

- $\mathbf x$：一条数据的特征；
- $\mathbf w,b$：模型需要学习的参数；
- $\hat y$：预测结果。


## 2. 导数的本质：局部变化率

平均变化率使用两个点：

$$
\frac{f(x+\Delta x)-f(x)}{\Delta x}.
$$

当 $\Delta x$ 越来越小时，两点之间的割线逐渐接近该点的切线。极限就是导数：

$$
f'(x)=\lim_{\Delta x\to 0}
\frac{f(x+\Delta x)-f(x)}{\Delta x}.
$$

直观解释：在当前点附近，$x$ 增加一个很小的量 $\Delta x$，输出大约改变

$$
\Delta y\approx f'(x)\Delta x.
$$

这叫做**局部线性近似**。神经网络虽然整体非常复杂，但反向传播正是在每个局部使用这种线性关系。


In [2]:
def f(x):
    return x ** 2


x0 = 2.0
true_slope = 2 * x0  # f'(x)=2x

for delta in [1.0, 0.5, 0.1, 0.01, 0.001]:
    approximate_slope = (f(x0 + delta) - f(x0)) / delta
    print(f"Δx={delta:<5} 近似斜率={approximate_slope:.4f}")

# linspace 在区间两端之间均匀生成 300 个点，用于画平滑曲线。
x_plot = np.linspace(-1, 4, 300)
tangent = f(x0) + true_slope * (x_plot - x0)

plt.figure(figsize=(8, 4))
plt.plot(x_plot, f(x_plot), label=r"$f(x)=x^2$")
plt.plot(x_plot, tangent, "--", label="x=2 处的切线")
plt.scatter([x0], [f(x0)], color="crimson", zorder=3)
plt.xlabel("x")
plt.ylabel("y")
plt.title("导数就是函数图像在某一点的切线斜率")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


Δx=1.0   近似斜率=5.0000
Δx=0.5   近似斜率=4.5000
Δx=0.1   近似斜率=4.1000
Δx=0.01  近似斜率=4.0100
Δx=0.001 近似斜率=4.0010


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2478212886.py:25: UserWarning: Glyph 23548 (\N{CJK UNIFIED IDEOGRAPH-5BFC}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2478212886.py:25: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2478212886.py:25: UserWarning: Glyph 23601 (\N{CJK UNIFIED IDEOGRAPH-5C31}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2478212886.py:25: UserWarning: Glyph 26159 (\N{CJK UNIFIED IDEOGRAPH-662F}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2478212886.py:25: UserWarning: Glyph 20989 (\N{CJK UNIFIED IDEOGRAPH-51FD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhy

### 导数正负代表什么？

- $f'(x)>0$：向右走一点，函数值增大；
- $f'(x)<0$：向右走一点，函数值减小；
- $f'(x)=0$：局部是水平的，但不一定是最低点，也可能是最高点或鞍点。

导数的绝对值越大，局部变化越快。


## 3. 常见求导规则

| 函数 | 导数 |
|---|---|
| 常数 $c$ | $0$ |
| $x^n$ | $nx^{n-1}$ |
| $e^x$ | $e^x$ |
| $\log x$ | $1/x$ |
| $\sigma(x)=1/(1+e^{-x})$ | $\sigma(x)(1-\sigma(x))$ |

组合规则：

$$
(u+v)'=u'+v'
$$

$$
(uv)'=u'v+uv'
$$

$$
\left(\frac uv\right)'=\frac{u'v-uv'}{v^2}
$$

不必一次背完。机器学习中最重要的是平方、指数、对数和链式法则。


## 4. 单变量链式法则

如果

$$
y=f(u),\qquad u=g(x),
$$

那么

$$
\frac{dy}{dx}
=
\frac{dy}{du}
\frac{du}{dx}.
$$

例：$y=(3x+1)^2$。令 $u=3x+1$，则 $y=u^2$：

$$
\frac{dy}{du}=2u,\qquad
\frac{du}{dx}=3,
$$

所以

$$
\frac{dy}{dx}=2(3x+1)\cdot 3=6(3x+1).
$$

直观上，每一层都在放大或缩小变化量，总影响是各层局部变化率的乘积。


In [3]:
def composed_function(x):
    return (3 * x + 1) ** 2


x0 = 2.0
analytical = 6 * (3 * x0 + 1)
epsilon = 1e-5
numerical = (
    composed_function(x0 + epsilon)
    - composed_function(x0 - epsilon)
) / (2 * epsilon)

print("解析导数:", analytical)
print("数值导数:", numerical)


解析导数: 42.0
数值导数: 42.000000000896875


# 第二部分：多变量函数与偏导数

## 5. 为什么机器学习需要多元微积分？

一个模型通常有很多参数：

$$
J(w_1,w_2,\dots,w_n,b).
$$

损失 $J$ 同时由所有参数决定。训练需要回答：

- 只改变 $w_1$ 会怎样？
- 只改变 $w_2$ 会怎样？
- 所有参数应该一起往哪个方向移动？

偏导数回答前两个问题，梯度回答第三个问题。


## 6. 二元函数：曲面与等高线

例：

$$
f(x,y)=x^2+2y^2.
$$

输入是平面上的点 $(x,y)$，输出是高度。它的三维图像像一个碗。

等高线把具有相同函数值的点画在同一条线上，类似地图上的海拔线。机器学习经常使用等高线观察损失函数。


In [4]:
x = np.linspace(-3, 3, 160)
y = np.linspace(-3, 3, 160)
# meshgrid 把两个一维坐标轴扩展成二维坐标网格。
xx, yy = np.meshgrid(x, y)
zz = xx ** 2 + 2 * yy ** 2

fig = plt.figure(figsize=(11, 4))
ax3d = fig.add_subplot(1, 2, 1, projection="3d")
ax3d.plot_surface(xx, yy, zz, cmap="viridis", alpha=0.9)
ax3d.set(title=r"曲面 $f(x,y)=x^2+2y^2$", xlabel="x", ylabel="y")

ax2d = fig.add_subplot(1, 2, 2)
# contour 把相同函数值连接成等高线；levels 控制层数。
contour = ax2d.contour(xx, yy, zz, levels=15)
ax2d.clabel(contour, fontsize=7)
ax2d.set(title="同一个函数的等高线", xlabel="x", ylabel="y")
ax2d.axis("equal")

plt.tight_layout()
plt.show()


Font 'default' does not have a glyph for '\u66f2' [U+66f2], substituting with a dummy symbol.


Font 'default' does not have a glyph for '\u9762' [U+9762], substituting with a dummy symbol.


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/216550518.py:19: UserWarning: Glyph 21516 (\N{CJK UNIFIED IDEOGRAPH-540C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/216550518.py:19: UserWarning: Glyph 19968 (\N{CJK UNIFIED IDEOGRAPH-4E00}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/216550518.py:19: UserWarning: Glyph 20010 (\N{CJK UNIFIED IDEOGRAPH-4E2A}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/216550518.py:19: UserWarning: Glyph 20989 (\N{CJK UNIFIED IDEOGRAPH-51FD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/216550518.py:19: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317

## 7. 偏导数：一次只让一个变量变化

对 $f(x,y)=x^2+2y^2$：

$$
\frac{\partial f}{\partial x}=2x,
\qquad
\frac{\partial f}{\partial y}=4y.
$$

求 $\partial f/\partial x$ 时，把 $y$ 暂时看作常数；求 $\partial f/\partial y$ 时，把 $x$ 暂时看作常数。

在点 $(1,2)$：

$$
\frac{\partial f}{\partial x}=2,\qquad
\frac{\partial f}{\partial y}=8.
$$

含义：在这个点附近，$y$ 方向的单位变化对输出影响更大。


In [5]:
def f2(point):
    x, y = point
    return x ** 2 + 2 * y ** 2


point = np.array([1.0, 2.0])
analytical_partial_x = 2 * point[0]
analytical_partial_y = 4 * point[1]

epsilon = 1e-5
unit_x = np.array([1.0, 0.0])
unit_y = np.array([0.0, 1.0])

numerical_partial_x = (
    f2(point + epsilon * unit_x) - f2(point - epsilon * unit_x)
) / (2 * epsilon)
numerical_partial_y = (
    f2(point + epsilon * unit_y) - f2(point - epsilon * unit_y)
) / (2 * epsilon)

print("∂f/∂x:", analytical_partial_x, numerical_partial_x)
print("∂f/∂y:", analytical_partial_y, numerical_partial_y)


∂f/∂x: 2.0 2.0000000000131024
∂f/∂y: 8.0 8.00000000005241


## 8. 全微分：多个变量同时变化

当 $x$ 和 $y$ 都有很小变化时：

$$
\Delta f
\approx
\frac{\partial f}{\partial x}\Delta x
+
\frac{\partial f}{\partial y}\Delta y.
$$

写成向量形式：

$$
\Delta f\approx \nabla f^T\Delta\mathbf{x}.
$$

这表示：总变化约等于“每个输入的变化 × 该输入的敏感度”之和。反向传播得到的梯度，本质上就是这些敏感度。


# 第三部分：梯度与梯度下降

## 9. 梯度是什么？

梯度把所有偏导数组成一个向量：

$$
\nabla f(x,y)=
\begin{bmatrix}
\partial f/\partial x\\
\partial f/\partial y
\end{bmatrix}.
$$

对 $f(x,y)=x^2+2y^2$：

$$
\nabla f(x,y)=
\begin{bmatrix}
2x\\
4y
\end{bmatrix}.
$$

梯度有两个重要性质：

1. 它指向函数值上升最快的方向；
2. 它垂直于当前位置的等高线。

因此，负梯度 $-\nabla f$ 指向局部下降最快的方向。


In [6]:
grid = np.linspace(-2.5, 2.5, 17)
xx, yy = np.meshgrid(grid, grid)
zz = xx ** 2 + 2 * yy ** 2
grad_x = 2 * xx
grad_y = 4 * yy

# 只显示方向，避免远处箭头过长。
length = np.sqrt(grad_x ** 2 + grad_y ** 2)
safe_length = np.where(length == 0, 1, length)

plt.figure(figsize=(6, 5))
plt.contour(xx, yy, zz, levels=12, alpha=0.65)
plt.quiver(
    xx, yy,
    grad_x / safe_length,
    grad_y / safe_length,
    color="crimson",
    alpha=0.75,
)
plt.xlabel("x")
plt.ylabel("y")
plt.title("梯度垂直于等高线，并指向上升最快方向")
plt.axis("equal")
plt.tight_layout()
plt.show()


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/3901725036.py:24: UserWarning: Glyph 26799 (\N{CJK UNIFIED IDEOGRAPH-68AF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/3901725036.py:24: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGRAPH-5EA6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/3901725036.py:24: UserWarning: Glyph 22402 (\N{CJK UNIFIED IDEOGRAPH-5782}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/3901725036.py:24: UserWarning: Glyph 30452 (\N{CJK UNIFIED IDEOGRAPH-76F4}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/3901725036.py:24: UserWarning: Glyph 20110 (\N{CJK UNIFIED IDEOGRAPH-4E8E}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhy

## 10. 方向导数：沿任意方向的变化率

若 $\mathbf u$ 是单位向量，沿 $\mathbf u$ 的方向导数为：

$$
D_{\mathbf u}f=\nabla f^T\mathbf u.
$$

它是梯度和方向向量的点积。根据点积的几何意义：

$$
\nabla f^T\mathbf u
=
\|\nabla f\|\|\mathbf u\|\cos\theta.
$$

当 $\mathbf u$ 与梯度同方向时，$\cos\theta=1$，上升最快；与负梯度同方向时下降最快。


In [7]:
point = np.array([1.0, 2.0])
gradient = np.array([2 * point[0], 4 * point[1]])

direction = np.array([1.0, 1.0])
# norm 计算向量长度；除以长度后得到单位方向向量。
direction = direction / np.linalg.norm(direction)  # 转成单位向量

directional_derivative = gradient @ direction
print("梯度:", gradient)
print("单位方向:", direction)
print("沿该方向的变化率:", directional_derivative)


梯度: [2. 8.]
单位方向: [0.7071 0.7071]
沿该方向的变化率: 7.071067811865475


## 11. 梯度下降

为了最小化函数，参数沿负梯度方向更新：

$$
\boldsymbol\theta
\leftarrow
\boldsymbol\theta-\alpha\nabla J(\boldsymbol\theta).
$$

- $\boldsymbol\theta$：所有参数组成的向量；
- $J$：损失函数；
- $\alpha$：学习率；
- $\nabla J$：每个参数应如何影响损失。

学习率太小会很慢；太大会越过谷底甚至发散。梯度给出方向和相对敏感度，学习率控制整体步长。


In [8]:
def objective(theta):
    x, y = theta
    return x ** 2 + 2 * y ** 2


def gradient_of_objective(theta):
    x, y = theta
    return np.array([2 * x, 4 * y])


theta = np.array([2.5, 2.0])
learning_rate = 0.12
path = [theta.copy()]

for _ in range(25):
    theta = theta - learning_rate * gradient_of_objective(theta)
    path.append(theta.copy())

path = np.array(path)

x = np.linspace(-3, 3, 200)
y = np.linspace(-3, 3, 200)
xx, yy = np.meshgrid(x, y)
zz = xx ** 2 + 2 * yy ** 2

plt.figure(figsize=(6, 5))
plt.contour(xx, yy, zz, levels=18)
plt.plot(path[:, 0], path[:, 1], "o-", color="crimson", markersize=3)
plt.scatter([0], [0], color="black", label="minimum")
plt.xlabel("parameter 1")
plt.ylabel("parameter 2")
plt.title("梯度下降在等高线上的路径")
plt.legend()
plt.axis("equal")
plt.tight_layout()
plt.show()

print("最终参数:", theta)
print("最终函数值:", objective(theta))


最终参数: [0.0026 0.    ]
最终函数值: 6.8637212023089126e-06


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2862494796.py:35: UserWarning: Glyph 26799 (\N{CJK UNIFIED IDEOGRAPH-68AF}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2862494796.py:35: UserWarning: Glyph 24230 (\N{CJK UNIFIED IDEOGRAPH-5EA6}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2862494796.py:35: UserWarning: Glyph 19979 (\N{CJK UNIFIED IDEOGRAPH-4E0B}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2862494796.py:35: UserWarning: Glyph 38477 (\N{CJK UNIFIED IDEOGRAPH-964D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/2862494796.py:35: UserWarning: Glyph 22312 (\N{CJK UNIFIED IDEOGRAPH-5728}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhy

## 12. 一个机器学习中的完整链式法则

单样本线性回归：

$$
\hat y=wx+b,\qquad
L=\frac12(\hat y-y)^2.
$$

计算图：

`w, x → 相乘 → 加 b → ŷ → 与 y 比较 → L`

对 $w$：

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial \hat y}
\frac{\partial \hat y}{\partial w}
=
(\hat y-y)x.
$$

对 $b$：

$$
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial \hat y}
\frac{\partial \hat y}{\partial b}
=
\hat y-y.
$$

最关键的直觉：

- $\hat y-y$：预测误差从损失传回预测值；
- 再乘 $x$：同一个预测误差对权重的影响由输入大小调节；
- 一个参数若对输出影响很大，它得到的梯度也会更大。


In [9]:
x = 3.0
y_true = 10.0
w = 2.0
b = 1.0

y_hat = w * x + b                   # 7
loss = 0.5 * (y_hat - y_true) ** 2 # 4.5

dloss_dyhat = y_hat - y_true        # -3
dyhat_dw = x                        # 3
dyhat_db = 1                        # 1

dloss_dw = dloss_dyhat * dyhat_dw   # -9
dloss_db = dloss_dyhat * dyhat_db   # -3

print("prediction:", y_hat, "loss:", loss)
print("dL/dw:", dloss_dw, "dL/db:", dloss_db)


prediction: 7.0 loss: 4.5
dL/dw: -9.0 dL/db: -3.0


# 第四部分：Jacobian、反向传播与 Hessian

## 13. 为什么标量函数用梯度，向量函数用 Jacobian？

如果 $f:\mathbb R^n\to\mathbb R$，多个输入、一个输出，全部偏导数可组成梯度。

如果 $\mathbf f:\mathbb R^n\to\mathbb R^m$，多个输入、多个输出，需要记录每个输出对每个输入的偏导数：

$$
J_{\mathbf f}=
\begin{bmatrix}
\partial f_1/\partial x_1 & \cdots & \partial f_1/\partial x_n\\
\vdots & \ddots & \vdots\\
\partial f_m/\partial x_1 & \cdots & \partial f_m/\partial x_n
\end{bmatrix}.
$$

Jacobian 的 shape 是 `(输出维度, 输入维度)`。


### Jacobian 例子

$$
\mathbf f(x,y)=
\begin{bmatrix}
x^2+y\\
xy
\end{bmatrix}.
$$

它有两个输入、两个输出，所以 Jacobian 是 $2\times2$：

$$
J_{\mathbf f}(x,y)=
\begin{bmatrix}
2x & 1\\
y & x
\end{bmatrix}.
$$

在 $(x,y)=(2,3)$：

$$
J_{\mathbf f}=
\begin{bmatrix}
4&1\\
3&2
\end{bmatrix}.
$$


In [10]:
x, y = 2.0, 3.0
jacobian = np.array([
    [2 * x, 1],
    [y, x],
])
print(jacobian)
print("Jacobian shape:", jacobian.shape)


[[4. 1.]
 [3. 2.]]
Jacobian shape: (2, 2)


## 14. 反向传播为什么不直接构造完整 Jacobian？

神经网络每层可能有数百万个输入和输出，完整 Jacobian 会非常大。训练真正需要的是“最终标量损失对参数的梯度”，因此反向模式逐层计算 vector-Jacobian product（VJP）：

$$
\mathbf v^T J.
$$

这里 $\mathbf v$ 是从后面一层传回来的上游梯度。这样只计算最终需要的组合，而不是保存完整 Jacobian。

反向传播可以理解为：

1. 前向传播保存中间值；
2. 从损失的梯度 1 开始；
3. 每经过一个操作，就乘上该操作的局部导数；
4. 多条路径汇合时，把梯度贡献相加。


## 15. Hessian：梯度如何变化？

一阶导数描述函数的斜率；二阶导数描述斜率的变化，也就是曲率。

多变量函数的二阶偏导数组成 Hessian：

$$
H=
\begin{bmatrix}
\partial^2f/\partial x_1^2 &
\partial^2f/\partial x_1\partial x_2\\
\partial^2f/\partial x_2\partial x_1 &
\partial^2f/\partial x_2^2
\end{bmatrix}.
$$

对 $f(x,y)=x^2+2y^2$：

$$
H=
\begin{bmatrix}
2&0\\
0&4
\end{bmatrix}.
$$

两个方向都向上弯，是凸碗。若某些方向向上、某些方向向下，就可能出现鞍点。

初学阶段不必计算 Hessian，但要知道它解释了：

- 为什么某些方向很陡、某些方向很平；
- 为什么特征缩放能改善梯度下降；
- 为什么学习率太大会在陡峭方向震荡。


In [11]:
x = np.linspace(-2, 2, 120)
y = np.linspace(-2, 2, 120)
xx, yy = np.meshgrid(x, y)

convex = xx ** 2 + yy ** 2
saddle = xx ** 2 - yy ** 2

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.plot_surface(xx, yy, convex, cmap="viridis")
ax1.set_title("凸函数：所有方向向上弯")

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(xx, yy, saddle, cmap="coolwarm")
ax2.set_title("鞍点：不同方向曲率符号不同")

plt.tight_layout()
plt.show()


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 20984 (\N{CJK UNIFIED IDEOGRAPH-51F8}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 20989 (\N{CJK UNIFIED IDEOGRAPH-51FD}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 25968 (\N{CJK UNIFIED IDEOGRAPH-6570}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 65306 (\N{FULLWIDTH COLON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 25152 (\N{CJK UNIFIED IDEOGRAPH-6240}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3s

/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 38797 (\N{CJK UNIFIED IDEOGRAPH-978D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 28857 (\N{CJK UNIFIED IDEOGRAPH-70B9}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 19981 (\N{CJK UNIFIED IDEOGRAPH-4E0D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 21516 (\N{CJK UNIFIED IDEOGRAPH-540C}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_68286/1893598622.py:17: UserWarning: Glyph 26354 (\N{CJK UNIFIED IDEOGRAPH-66F2}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/var/folders/fs/nqhy

# 第五部分：矩阵形式的机器学习梯度

## 16. 多元线性回归的梯度

对 $m$ 个样本、$n$ 个特征：

$$
X\in\mathbb R^{m\times n},\quad
\mathbf w\in\mathbb R^n,\quad
\mathbf y\in\mathbb R^m.
$$

预测：

$$
\hat{\mathbf y}=X\mathbf w+b.
$$

误差向量：

$$
\mathbf e=\hat{\mathbf y}-\mathbf y.
$$

均方误差目标：

$$
J=\frac{1}{2m}\mathbf e^T\mathbf e.
$$

最终梯度：

$$
\nabla_{\mathbf w}J=\frac1mX^T\mathbf e,
\qquad
\frac{\partial J}{\partial b}=\frac1m\sum_i e_i.
$$

### 为什么是 $X^T\mathbf e$？

每个权重 $w_j$ 只与第 $j$ 个特征列相关。$X^T$ 的第 $j$ 行就是该特征在所有样本上的值；它与误差向量做点积，得到该特征对全部误差的总贡献。

shape 检查：

$$
(n,m)@(m,)\to(n,).
$$

结果刚好与 $\mathbf w$ 同形状。


In [12]:
X = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
])                              # (m=3, n=2)
y = np.array([5.0, 4.0, 10.0]) # (m,)
w = np.array([1.0, 1.0])       # (n,)
b = 0.0

predictions = X @ w + b
errors = predictions - y
dw = X.T @ errors / len(y)
db = errors.mean()

print("predictions:", predictions)
print("errors:", errors)
print("dw:", dw, "shape:", dw.shape)
print("db:", db)


predictions: [3. 3. 7.]
errors: [-2. -1. -3.]
dw: [-4.3333 -5.6667] shape: (2,)
db: -2.0


## 17. Logistic Regression 中的神奇消去

$$
z=\mathbf w^T\mathbf x+b,\qquad
p=\sigma(z).
$$

二元交叉熵：

$$
L=-y\log p-(1-y)\log(1-p).
$$

两个局部导数：

$$
\frac{\partial L}{\partial p}
=
-\frac yp+\frac{1-y}{1-p},
$$

$$
\frac{\partial p}{\partial z}=p(1-p).
$$

根据链式法则：

$$
\frac{\partial L}{\partial z}
=
\frac{\partial L}{\partial p}
\frac{\partial p}{\partial z}
=p-y.
$$

因此：

$$
\frac{\partial L}{\partial\mathbf w}
=(p-y)\mathbf x.
$$

这就是为什么逻辑回归最终梯度的形式与线性回归相似：都包含“预测减真实值”，但两者的模型和损失函数并不相同。


## 18. L2 正则化的梯度

在原损失后增加：

$$
\frac{\lambda}{2m}\|\mathbf w\|_2^2
=
\frac{\lambda}{2m}\sum_jw_j^2.
$$

对 $\mathbf w$ 求导：

$$
\nabla_{\mathbf w}
\frac{\lambda}{2m}\|\mathbf w\|_2^2
=
\frac{\lambda}{m}\mathbf w.
$$

所以更新时会额外减去与当前权重成比例的一项，使权重不断缩小，这也是“weight decay”名称的来源。


# 第六部分：数值检查与学习建议

## 19. 数值梯度检查

中心差分：

$$
\frac{\partial f}{\partial x_j}
\approx
\frac{f(\mathbf x+\epsilon\mathbf e_j)
-f(\mathbf x-\epsilon\mathbf e_j)}
{2\epsilon}.
$$

数值梯度很慢，但适合检查手推梯度或自定义反向传播。若解析梯度和数值梯度差很多，优先检查：

1. 正负号；
2. 是否漏掉平均中的 $1/m$；
3. shape 和广播；
4. 链式法则是否漏乘某一层；
5. 正则化项是否包含或排除了 bias。


In [13]:
def numerical_gradient(function, point, epsilon=1e-5):
    gradient = np.zeros_like(point, dtype=float)

    for index in range(len(point)):
        step = np.zeros_like(point, dtype=float)
        step[index] = epsilon
        gradient[index] = (
            function(point + step) - function(point - step)
        ) / (2 * epsilon)

    return gradient


point = np.array([1.0, 2.0])
analytical = np.array([2 * point[0], 4 * point[1]])
numerical = numerical_gradient(f2, point)

print("解析梯度:", analytical)
print("数值梯度:", numerical)
# allclose 用容差比较浮点数组，避免直接使用 ==。
print("是否接近:", np.allclose(analytical, numerical))


解析梯度: [2. 8.]
数值梯度: [2. 8.]
是否接近: True


## 20. 初学者学习检查表

看到一个求导问题时，按顺序做：

1. 写清输入和输出；
2. 把复杂表达式拆成中间变量；
3. 画出计算顺序；
4. 写每一步的局部导数；
5. 从输出向输入应用链式法则；
6. 多条路径的梯度相加；
7. 检查每个梯度的 shape；
8. 用数值梯度验证。

## 练习

1. 对 $f(x)=x^3-2x$ 求导，并画出函数和 $x=1$ 处切线。
2. 对 $f(x,y)=3x^2+xy+y^2$ 求两个偏导数。
3. 在点 $(1,2)$ 计算上一题的梯度。
4. 写出 $z=wx+b,\ p=\sigma(z),\ L=(p-y)^2$ 对 $w$ 的链式法则。
5. 推导两层标量网络 $h=wx,\ \hat y=vh,\ L=(\hat y-y)^2/2$ 对 $w,v$ 的梯度。
6. 修改梯度下降学习率，观察收敛、震荡和发散。

> 不需要一次掌握所有符号。先能用语言解释“一个量改变会怎样影响下一个量”，公式会逐渐变得自然。
